# Eda Operation Ablation (MiniLM-L6-H384-uncased)


In [ ]:
# Cell 1: Setup Environment and GPU
import subprocess, sys

# Check GPU
try:
    gpu_name = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
    print("GPU Detected:", gpu_name)
    if "P100" in gpu_name:
        print("Installing specific PyTorch version for P100...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", 
                               "--no-deps", "torch==2.5.1", "--index-url", "https://download.pytorch.org/whl/cu121"])
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"], 
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
except Exception as e:
    print("Failed to detect GPU. Please ensure GPU T4 x2 or P100 is enabled in Notebook Settings.")

# Install dependencies
print("Installing required packages...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.48,<5", "sentencepiece", "nltk"])
print("[OK] Environment setup complete.")


In [ ]:
# Cell 2: Kaggle Path Setup
import sys
from pathlib import Path

# Add dataset source to path (to load cyber_eda.py and generic_eda.py)
kaggle_inputs = list(Path("/kaggle/input").rglob("src/cyber_eda.py"))
if kaggle_inputs:
    project_root = str(kaggle_inputs[0].parent.parent)
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    print(f"Added {project_root} to sys.path")
else:
    print("Warning: Could not automatically find the dataset folder in /kaggle/input")
    print("Make sure you have added the dataset containing 'src' and 'dataset' folders to this notebook.")


## Self-Contained Trainer
Model: ELECTRA-Small | HuggingFace Head | Threshold 0.5


In [ ]:
# Cell 3: Proxy Trainer (MiniLM-L6 + Mean Pooling + ASL)
import gc, random, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, precision_score, recall_score

class CTIDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.float)
        }

def compute_metrics(y_true, y_pred):
    return {
        "Macro_Precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "Macro_Recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "Macro_F1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "Micro_F1": float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
        "Micro_Precision": float(precision_score(y_true, y_pred, average="micro", zero_division=0)),
        "Micro_Recall": float(recall_score(y_true, y_pred, average="micro", zero_division=0))
    }

def analyze_4_tier(y_true, y_pred, train_label_counts, labels_list):
    from sklearn.metrics import precision_score, recall_score, f1_score
    import numpy as np
    
    prec_per = precision_score(y_true, y_pred, average=None, zero_division=0)
    rec_per  = recall_score(y_true, y_pred, average=None, zero_division=0)
    f1_per   = f1_score(y_true, y_pred, average=None, zero_division=0)
    
    tier_defs = {
        "Major (100-499)": lambda c: 100 <= c < 500,
        "Medium (30-99)": lambda c: 30 <= c < 100,
        "Tail (<30)": lambda c: c < 30
    }
    tier_summary   = {}
    zero_f1_labels = [lbl for i, lbl in enumerate(labels_list) if f1_per[i] == 0.0]
    for name, fn in tier_defs.items():
        idx = [i for i, lbl in enumerate(labels_list) if fn(train_label_counts.get(lbl, 0))]
        if idx:
            tier_summary[name] = {
                "num_labels": len(idx),
                "precision": float(np.mean(prec_per[idx])),
                "recall": float(np.mean(rec_per[idx])),
                "f1_score": float(np.mean(f1_per[idx])),
                "zero_f1_count": int(sum(1 for i in idx if f1_per[i] == 0.0))
            }
        else:
            tier_summary[name] = {"num_labels": 0, "precision": 0.0, "recall": 0.0, "f1_score": 0.0, "zero_f1_count": 0}
    return tier_summary, zero_f1_labels

PROXY_CONFIG = {
    "model_checkpoint": "nreimers/MiniLM-L6-H384-uncased",
    "max_length":        384,
    "epochs":            3,
    "train_batch_size":  16,
    "eval_batch_size":   32,
    "learning_rate":     1e-4,
    "weight_decay":      0.01,
    "warmup_ratio":      0.10,
    "max_grad_norm":     1.0,
    "dropout":           0.1,
    "threshold":         0.5,
    "fp16":              True,
    "num_workers":       2,
    "seed":              42,
    "gamma_neg":         4.0,
    "gamma_pos":         1.0,
    "asl_clip":          0.05,
    "asl_eps":           1e-8,
}

def masked_mean(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(last_hidden.dtype)
    return (last_hidden * mask).sum(1) / mask.sum(1).clamp_min(1e-9)

class ProxyClassifier(nn.Module):
    def __init__(self, checkpoint, num_labels, dropout_prob):
        super().__init__()
        model_config = AutoConfig.from_pretrained(checkpoint)
        self.encoder = AutoModel.from_pretrained(checkpoint, config=model_config)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(hidden, num_labels)
        
    def forward(self, input_ids, attention_mask, **kwargs):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        mean_pooled = masked_mean(out.last_hidden_state, attention_mask)
        logits = self.classifier(self.dropout(mean_pooled))
        return logits

class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4.0, gamma_pos=1.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits, targets):
        p = torch.sigmoid(logits)
        pos = p
        neg = 1.0 - p
        if self.clip is not None and self.clip > 0:
            neg = (neg + self.clip).clamp(max=1.0)
        loss = targets * torch.log(pos.clamp_min(self.eps)) + (1.0 - targets) * torch.log(neg.clamp_min(self.eps))
        pt = pos * targets + neg * (1.0 - targets)
        weight = torch.pow((1.0 - pt).clamp_min(0.0), self.gamma_pos * targets + self.gamma_neg * (1.0 - targets))
        return -(loss * weight).sum(1).mean()

def train_model(df_train, df_val, df_test, y_val, y_test, mlb, train_label_counts, labels_list, device, config=None):
    if config is None: config = PROXY_CONFIG
    seed = config.get("seed", 42)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

    y_train = mlb.transform(df_train["Labels"].apply(lambda x: str(x).split(","))).astype(np.float32)
    tokenizer  = AutoTokenizer.from_pretrained(config["model_checkpoint"])
    
    train_ds = CTIDataset(df_train["Cleaned_Text"].fillna("").tolist(), y_train, tokenizer, config["max_length"])
    val_ds   = CTIDataset(df_val["Cleaned_Text"].fillna("").tolist(),   y_val,   tokenizer, config["max_length"])
    test_ds  = CTIDataset(df_test["Cleaned_Text"].fillna("").tolist(),  y_test,  tokenizer, config["max_length"])

    pin = (device.type == "cuda")
    train_dl = DataLoader(train_ds, batch_size=config["train_batch_size"], shuffle=True,  num_workers=config["num_workers"], pin_memory=pin)
    val_dl   = DataLoader(val_ds,  batch_size=config["eval_batch_size"],  shuffle=False, num_workers=config["num_workers"], pin_memory=pin)
    test_dl  = DataLoader(test_ds, batch_size=config["eval_batch_size"],  shuffle=False, num_workers=config["num_workers"], pin_memory=pin)

    model = ProxyClassifier(
        config["model_checkpoint"], 
        num_labels=y_train.shape[1], 
        dropout_prob=config.get("dropout", 0.1)
    ).to(device)
    
    loss_fn = AsymmetricLoss(
        gamma_neg=config.get("gamma_neg", 4.0),
        gamma_pos=config.get("gamma_pos", 1.0),
        clip=config.get("asl_clip", 0.05),
        eps=config.get("asl_eps", 1e-8)
    )
    
    no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")
    optimizer_grouped_parameters = [
        {"params": [p for n, p in model.named_parameters() if p.requires_grad and not any(x in n for x in no_decay)],
         "weight_decay": config["weight_decay"]},
        {"params": [p for n, p in model.named_parameters() if p.requires_grad and any(x in n for x in no_decay)],
         "weight_decay": 0.0}
    ]
    optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=config["learning_rate"])
    total_steps  = len(train_dl) * config["epochs"]
    warmup_steps = int(total_steps * config["warmup_ratio"])
    scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler       = GradScaler() if config.get("fp16", True) and (device.type == "cuda") else None

    for epoch in range(1, config["epochs"] + 1):
        model.train()
        running_loss = 0.0
        for batch in train_dl:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels_b       = batch["labels"].to(device)
            optimizer.zero_grad(set_to_none=True)
            if scaler:
                with autocast():
                    logits = model(input_ids=input_ids, attention_mask=attention_mask)
                    loss = loss_fn(logits, labels_b)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config["max_grad_norm"])
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = loss_fn(logits, labels_b)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), config["max_grad_norm"])
                optimizer.step()
            scheduler.step()
            running_loss += loss.item()
        print(f"      epoch {epoch}/{config['epochs']} | loss={running_loss/len(train_dl):.4f}")

    def predict_probs(dl):
        model.eval()
        probs_list = []
        with torch.no_grad():
            for batch in dl:
                iids = batch["input_ids"].to(device)
                amsk = batch["attention_mask"].to(device)
                if scaler:
                    with autocast(): 
                        logits = model(input_ids=iids, attention_mask=amsk)
                else: 
                    logits = model(input_ids=iids, attention_mask=amsk)
                probs = torch.sigmoid(logits).cpu().numpy()
                probs_list.append(probs)
        return np.vstack(probs_list)

    val_probs = predict_probs(val_dl)
    test_probs = predict_probs(test_dl)
    
    # Fixed Threshold
    best_thresh = config.get("threshold", 0.5)
    print(f"      -> Using fixed threshold: {best_thresh:.2f}")

    y_val_pred = (val_probs >= best_thresh).astype(int)
    y_test_pred = (test_probs >= best_thresh).astype(int)

    val_metrics  = compute_metrics(y_val,  y_val_pred)
    test_metrics = compute_metrics(y_test, y_test_pred)
    val_tiers,  zv = analyze_4_tier(y_val,  y_val_pred,  train_label_counts, labels_list)
    test_tiers, zt = analyze_4_tier(y_test, y_test_pred, train_label_counts, labels_list)

    del model, optimizer, scheduler, scaler, train_dl, val_dl, test_dl, train_ds, val_ds, test_ds
    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return val_metrics, test_metrics, val_tiers, test_tiers, zv, zt

def setup_device_and_seed(seed=42):
    if not torch.cuda.is_available(): raise RuntimeError("GPU not available.")
    device = torch.device("cuda")
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    props = torch.cuda.get_device_properties(0)
    print(f"[GPU] {props.name} | VRAM: {props.total_memory / 1024**3:.1f} GB")
    return device

def resolve_paths(script_file="notebook.py"):
    IS_KAGGLE = Path("/kaggle/working").exists()
    if IS_KAGGLE:
        src_candidates = sorted(Path("/kaggle/input").rglob("src/cyber_eda.py"))
        if src_candidates:
            project_root = src_candidates[0].parent.parent
        else:
            tc = sorted(Path("/kaggle/input").glob("*/dataset/processed/joint/train.csv"))
            project_root = tc[0].parent.parent.parent.parent if tc else Path("/kaggle/input")
        data_base = project_root / "dataset" / "processed"
        if not data_base.exists() and (project_root / "processed").exists(): data_base = project_root / "processed"
        out_base  = Path("/kaggle/working") / "results" / "EDA ablation results"
    else:
        project_root = Path(".").resolve()
        data_base    = project_root / "dataset" / "processed"
        out_base     = project_root / "results" / "EDA ablation results"
    out_base.mkdir(parents=True, exist_ok=True)
    return project_root, data_base, out_base, IS_KAGGLE


## Ablation Logic


In [ ]:
"""
EDA Operation Ablation Study — MiniLM-L6 Proxy
===============================================
Determines which individual EDA operation (SR, RI, RS, RD) contributes most
to performance on the Joint CTI dataset.

Proxy model: nreimers/MiniLM-L6-H384-uncased
Dataset: joint only (same as ratio tuning)

IMPORTANT: Set OPTIMAL_ALPHA values below AFTER running ratio ablation studies:
  OPTIMAL_ALPHA["cyber"]   = result from cybereda_ratio_ablation_study.py
  OPTIMAL_ALPHA["generic"] = result from genericeda_ratio_ablation_study.py

Alternatively, pass via --alpha_cyber and --alpha_generic CLI arguments.

Configurations (11 total):
  A0           : Baseline -- no augmentation
  Cyber_SR/RI/RS/RD/Full : CyberEDA individual operations + full
  Generic_SR/RI/RS/RD/Full: GenericEDA individual operations + full

Output: results/EDA ablation results/operation_ablation/
"""

import os, sys, re, json, time, pickle, random, argparse, warnings
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path
from collections import Counter

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", line_buffering=True)
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", line_buffering=True)



from src.cyber_eda import build_cyber_knowledge_base, cyber_eda as apply_cyber_eda, single_eda as cyber_single_eda
from src.generic_eda import apply_generic_eda



# =============================================================================
# Optimal alpha -- UPDATE after ratio ablation
# These defaults will be overridden by --alpha_cyber / --alpha_generic args
# =============================================================================
OPTIMAL_ALPHA = {
    "cyber":   0.10,   # placeholder: replace with cybereda_optimal_alpha_joint.json value
    "generic": 0.10,   # placeholder: replace with genericeda_optimal_alpha_joint.json value
}


def try_load_optimal_alphas(out_base):  # hardcoded to 0.10
    """Auto-load alpha* from ratio ablation JSON files if they exist."""
    cyber_json   = out_base / "cybereda_ratio_ablation"  / "cybereda_optimal_alpha_joint.json"
    generic_json = out_base / "genericeda_ratio_ablation" / "genericeda_optimal_alpha_joint.json"
    if cyber_json.exists():
        with open(cyber_json) as f:
            OPTIMAL_ALPHA["cyber"] = json.load(f)["optimal_alpha"]
        print(f"  [AUTO] CyberEDA alpha*   = {OPTIMAL_ALPHA['cyber']} (from {cyber_json.name})")
    if generic_json.exists():
        with open(generic_json) as f:
            OPTIMAL_ALPHA["generic"] = json.load(f)["optimal_alpha"]
        print(f"  [AUTO] GenericEDA alpha* = {OPTIMAL_ALPHA['generic']} (from {generic_json.name})")


# =============================================================================
# Config builder
# =============================================================================
def build_configs(ca, ga):
    """11 ablation configurations: baseline + 5 Cyber + 5 Generic."""
    return [
        {"id": "A0",           "eda_type": None,      "sr": 0.0, "ri": 0.0, "rs": 0.0, "rd": 0.0,
         "label": "Baseline",          "group": "Baseline"},
        {"id": "Cyber_SR",     "eda_type": "cyber",   "sr": ca,  "ri": 0.0, "rs": 0.0, "rd": 0.0,
         "label": f"Cyber SR (a={ca})", "group": "CyberEDA"},
        {"id": "Cyber_RI",     "eda_type": "cyber",   "sr": 0.0, "ri": ca,  "rs": 0.0, "rd": 0.0,
         "label": f"Cyber RI (a={ca})", "group": "CyberEDA"},
        {"id": "Cyber_RS",     "eda_type": "cyber",   "sr": 0.0, "ri": 0.0, "rs": ca,  "rd": 0.0,
         "label": f"Cyber RS (a={ca})", "group": "CyberEDA"},
        {"id": "Cyber_RD",     "eda_type": "cyber",   "sr": 0.0, "ri": 0.0, "rs": 0.0, "rd": ca,
         "label": f"Cyber RD (a={ca})", "group": "CyberEDA"},
        {"id": "Cyber_Full",   "eda_type": "cyber",   "sr": ca,  "ri": ca,  "rs": ca,  "rd": ca,
         "label": f"Cyber Full",       "group": "CyberEDA"},
        {"id": "Generic_SR",   "eda_type": "generic", "sr": ga,  "ri": 0.0, "rs": 0.0, "rd": 0.0,
         "label": f"Generic SR (a={ga})", "group": "GenericEDA"},
        {"id": "Generic_RI",   "eda_type": "generic", "sr": 0.0, "ri": ga,  "rs": 0.0, "rd": 0.0,
         "label": f"Generic RI (a={ga})", "group": "GenericEDA"},
        {"id": "Generic_RS",   "eda_type": "generic", "sr": 0.0, "ri": 0.0, "rs": ga,  "rd": 0.0,
         "label": f"Generic RS (a={ga})", "group": "GenericEDA"},
        {"id": "Generic_RD",   "eda_type": "generic", "sr": 0.0, "ri": 0.0, "rs": 0.0, "rd": ga,
         "label": f"Generic RD (a={ga})", "group": "GenericEDA"},
        {"id": "Generic_Full", "eda_type": "generic", "sr": ga,  "ri": ga,  "rs": ga,  "rd": ga,
         "label": f"Generic Full",     "group": "GenericEDA"},
    ]


# =============================================================================
# Augmentation
# =============================================================================
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)


def generate_augmented_dataset(df_train, cfg, protected_set, synonym_dict,
                                target_count, seed=42):
    set_seed(seed)
    df = df_train.copy()
    if "is_augmented" not in df.columns:
        df["is_augmented"] = 0
    df["_lbl"] = df["Labels"].apply(lambda x: str(x).split(","))

    lc = Counter(lbl for sub in df["_lbl"] for lbl in sub)
    minority = {lbl: c for lbl, c in lc.items() if c < target_count}
    l2i = {}
    for loc_i, row in df.iterrows():
        for lbl in row["_lbl"]:
            l2i.setdefault(lbl, []).append(loc_i)

    rows_to_aug = []
    for lbl, count in minority.items():
        rows_to_aug.extend(random.choices(l2i[lbl], k=target_count - count))

    sr, ri, rs, rd = cfg["sr"], cfg["ri"], cfg["rs"], cfg["rd"]
    eda_type = cfg["eda_type"]
    augmented = []

    for seq_i, loc_i in enumerate(rows_to_aug):
        set_seed(seed + seq_i)
        row  = df.loc[loc_i]
        orig = str(row["Cleaned_Text"])

        if eda_type == "cyber":
            active = sum([sr > 0, ri > 0, rs > 0, rd > 0])
            if active > 1:
                aug = apply_cyber_eda(orig, protected_set, synonym_dict,
                                      alpha_sr=sr, alpha_ri=ri, alpha_rs=rs, p_rd=rd)
            elif sr > 0:
                aug = cyber_single_eda(orig, "sr", protected_set, synonym_dict, alpha_sr=sr)
            elif ri > 0:
                aug = cyber_single_eda(orig, "ri", protected_set, synonym_dict, alpha_ri=ri)
            elif rs > 0:
                aug = cyber_single_eda(orig, "rs", protected_set, synonym_dict, alpha_rs=rs)
            else:
                aug = cyber_single_eda(orig, "rd", protected_set, synonym_dict, p_rd=rd)
        elif eda_type == "generic":
            aug = apply_generic_eda(orig, alpha_sr=sr, alpha_ri=ri, alpha_rs=rs, p_rd=rd)
        else:
            aug = orig

        aug  = re.sub(r"\s+", " ", str(aug)).strip()
        toks = re.findall(r"[a-z0-9_\[\]]+(?:[./:-][a-z0-9_\[\]]+)*", aug.lower())
        augmented.append({
            "Cleaned_Text":    aug,
            "Labels":          row["Labels"],
            "Label_Count":     row["Label_Count"],
            "Tokenized_Text":  " ".join(toks),
            "source_sample_id": row["source_sample_id"],
            "is_augmented":    1,
        })

    return pd.concat([df.drop(columns=["_lbl"]), pd.DataFrame(augmented)], ignore_index=True)


# =============================================================================
# Visualization
# =============================================================================
CMAP = {"Baseline": "#607D8B", "CyberEDA": "#1565C0", "GenericEDA": "#E65100"}


def plot_macro_f1_comparison(results, out_dir, dataset_name="joint"):
    ids    = [r["config_id"] for r in results]
    labels = [r["label"]     for r in results]
    val_f1 = [r["val_metrics"]["Macro_F1"]  for r in results]
    tst_f1 = [r["test_metrics"]["Macro_F1"] for r in results]
    bv     = results[0]["val_metrics"]["Macro_F1"]
    bt     = results[0]["test_metrics"]["Macro_F1"]
    bar_colors = [CMAP[r["group"]] for r in results]

    x = np.arange(len(ids)); w = 0.38
    fig, ax = plt.subplots(figsize=(16, 6))
    ax.bar(x - w/2, val_f1, w, color=bar_colors, alpha=0.90, label="Val Macro-F1")
    ax.bar(x + w/2, tst_f1, w, color=bar_colors, alpha=0.50, hatch="//", label="Test Macro-F1")
    ax.axhline(bv, color="#37474F", ls="--", lw=1.2, label=f"Baseline Val  ({bv:.4f})")
    ax.axhline(bt, color="#455A64", ls=":",  lw=1.2, label=f"Baseline Test ({bt:.4f})")
    for xi, (v, t) in enumerate(zip(val_f1, tst_f1)):
        ax.text(xi-w/2, v+0.001, f"{v:.4f}", ha="center", va="bottom", fontsize=6.5, rotation=90)
        ax.text(xi+w/2, t+0.001, f"{t:.4f}", ha="center", va="bottom", fontsize=6.5, rotation=90)
    legend_patches = [
        Patch(facecolor=CMAP["Baseline"],  label="Baseline"),
        Patch(facecolor=CMAP["CyberEDA"],  label="CyberEDA (alias-based)"),
        Patch(facecolor=CMAP["GenericEDA"],label="GenericEDA (WordNet)"),
        Patch(facecolor="white", edgecolor="black", hatch="//", label="Test (hatched)"),
    ]
    ax.legend(handles=legend_patches, fontsize=9, loc="lower right")
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=40, ha="right", fontsize=8)
    ax.set_ylabel("Macro-F1 Score", fontsize=11)
    ax.set_ylim(min(val_f1+tst_f1)-0.03, max(val_f1+tst_f1)+0.06)
    ax.set_title(
        f"Figure 1: EDA Operation Ablation — {dataset_name.upper()} (MiniLM-L6 proxy)\n"
        "Solid: Validation | Hatched: Test | Blue: CyberEDA | Orange: GenericEDA", fontsize=11)
    ax.grid(axis="y", alpha=0.3); plt.tight_layout()
    p = out_dir / f"fig1_operation_ablation_macrof1_{dataset_name}.png"
    plt.savefig(p, dpi=150); plt.close(); print(f"  [SAVED] {p.name}")


def plot_delta_f1(results, out_dir, dataset_name="joint"):
    bv = results[0]["val_metrics"]["Macro_F1"]
    bt = results[0]["test_metrics"]["Macro_F1"]
    aug = results[1:]
    labels     = [r["label"] for r in aug]
    delta_val  = [r["val_metrics"]["Macro_F1"]  - bv for r in aug]
    delta_test = [r["test_metrics"]["Macro_F1"] - bt for r in aug]
    x = np.arange(len(aug)); w = 0.38
    vcols = ["#1565C0" if d >= 0 else "#C62828" for d in delta_val]
    tcols = ["#42A5F5" if d >= 0 else "#EF9A9A" for d in delta_test]
    fig, ax = plt.subplots(figsize=(15, 5.5))
    ax.bar(x - w/2, delta_val,  w, color=vcols, alpha=0.90, label="DeltaVal Macro-F1")
    ax.bar(x + w/2, delta_test, w, color=tcols, alpha=0.65, hatch="//", label="DeltaTest Macro-F1")
    ax.axhline(0, color="black", lw=1.0)
    for xi, (dv, dt) in enumerate(zip(delta_val, delta_test)):
        ax.text(xi-w/2, dv+(0.001 if dv >= 0 else -0.003),
                f"{dv:+.4f}", ha="center", va="bottom" if dv >= 0 else "top", fontsize=7, rotation=90)
        ax.text(xi+w/2, dt+(0.001 if dt >= 0 else -0.003),
                f"{dt:+.4f}", ha="center", va="bottom" if dt >= 0 else "top", fontsize=7, rotation=90)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=40, ha="right", fontsize=8)
    ax.set_ylabel("Delta Macro-F1 vs Baseline", fontsize=11)
    ax.set_title(
        f"Figure 2: Delta Macro-F1 per Operation — {dataset_name.upper()} (MiniLM-L6)\n"
        "Positive (Blue) = helpful | Negative (Red) = harmful", fontsize=11)
    ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3); plt.tight_layout()
    p = out_dir / f"fig2_delta_f1_{dataset_name}.png"
    plt.savefig(p, dpi=150); plt.close(); print(f"  [SAVED] {p.name}")


def plot_tail_f1(results, out_dir, dataset_name="joint"):
    labels    = [r["label"] for r in results]
    tail_val  = [r["val_tier_summary"].get("Tail (<30)", {}).get("f1_score", 0) for r in results]
    tail_test = [r["test_tier_summary"].get("Tail (<30)", {}).get("f1_score", 0) for r in results]
    bar_colors = [CMAP[r["group"]] for r in results]
    x = np.arange(len(results)); w = 0.38
    fig, ax = plt.subplots(figsize=(16, 5.5))
    ax.bar(x - w/2, tail_val,  w, color=bar_colors, alpha=0.90, label="Val Tail F1")
    ax.bar(x + w/2, tail_test, w, color=bar_colors, alpha=0.50, hatch="//", label="Test Tail F1")
    for xi, (v, t) in enumerate(zip(tail_val, tail_test)):
        ax.text(xi-w/2, v+0.001, f"{v:.4f}", ha="center", va="bottom", fontsize=6.5, rotation=90)
        ax.text(xi+w/2, t+0.001, f"{t:.4f}", ha="center", va="bottom", fontsize=6.5, rotation=90)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=40, ha="right", fontsize=8)
    ax.set_ylabel("Mean F1 — Tail Classes (<30 train samples)", fontsize=11)
    ax.set_title(
        f"Figure 3: Rare-Class (Tail) F1 per Operation — {dataset_name.upper()} (MiniLM-L6)\n"
        "Key evidence: which operation improves recognition of underrepresented ATT&CK techniques",
        fontsize=11)
    ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3); plt.tight_layout()
    p = out_dir / f"fig3_rare_class_f1_{dataset_name}.png"
    plt.savefig(p, dpi=150); plt.close(); print(f"  [SAVED] {p.name}")


def print_summary_table(results):
    bv = results[0]["val_metrics"]["Macro_F1"]
    bt = results[0]["test_metrics"]["Macro_F1"]
    line = "=" * 120
    hdr  = (f"  {'Config':<16} {'Group':<12} {'Val F1':>10} {'Test F1':>10} "
            f"{'DVal':>8} {'DTest':>8} {'ZF1-V':>8} {'ZF1-T':>8} "
            f"{'Tail-V':>8} {'Tail-T':>8}")
    print(f"\n{line}\n  EDA OPERATION ABLATION — MiniLM-L6 PROXY — RESULTS\n{hdr}\n{'-'*120}")
    for r in results:
        dv   = r["val_metrics"]["Macro_F1"]  - bv
        dt   = r["test_metrics"]["Macro_F1"] - bt
        tv   = r["val_tier_summary"].get("Tail (<30)", {}).get("f1_score", 0)
        tt   = r["test_tier_summary"].get("Tail (<30)", {}).get("f1_score", 0)
        mark = "  <- BASELINE" if r["config_id"] == "A0" else ""
        print(f"  {r['config_id']:<16} {r['group']:<12} "
              f"{r['val_metrics']['Macro_F1']:>10.4f} {r['test_metrics']['Macro_F1']:>10.4f} "
              f"{dv:>+8.4f} {dt:>+8.4f} {r['num_zero_f1_val']:>8} {r['num_zero_f1_test']:>8} "
              f"{tv:>8.4f} {tt:>8.4f}{mark}")
    print(line)


def load_checkpoint(path):
    if path.exists():
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    return []


def save_checkpoint(path, results):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)


# =============================================================================
# Main
# =============================================================================
def run_operation_ablation(target_dataset="joint", seed=42,
                           alpha_cyber=None, alpha_generic=None):
    project_root, data_base, out_base, IS_KAGGLE = resolve_paths('notebook.py')
    data_dir = data_base / target_dataset
    out_dir  = out_base / "operation_ablation"
    out_dir.mkdir(parents=True, exist_ok=True)

    # Auto-load optimal alphas from ratio ablation results
    # try_load_optimal_alphas(out_base)  # hardcoded to 0.10

    # CLI overrides take highest priority
    if alpha_cyber   is not None: OPTIMAL_ALPHA["cyber"]   = alpha_cyber
    if alpha_generic is not None: OPTIMAL_ALPHA["generic"] = alpha_generic

    ca = OPTIMAL_ALPHA["cyber"]
    ga = OPTIMAL_ALPHA["generic"]

    print("=" * 80)
    print(f"  EDA OPERATION ABLATION — {target_dataset.upper()}")
    print(f"  Proxy: nreimers/MiniLM-L6-H384-uncased | CyberEDA alpha={ca} | GenericEDA alpha={ga}")
    print("=" * 80)

    device = setup_device_and_seed(seed)

    for fname in ["train.csv", "val.csv", "test.csv", "multilabel_binarizer.pkl"]:
        if not (data_dir / fname).exists():
            raise FileNotFoundError(f"Missing: {data_dir / fname}")

    df_train = pd.read_csv(data_dir / "train.csv")
    df_val   = pd.read_csv(data_dir / "val.csv")
    df_test  = pd.read_csv(data_dir / "test.csv")

    with open(data_dir / "multilabel_binarizer.pkl", "rb") as fh:
        mlb = pickle.load(fh)

    labels_list = list(mlb.classes_)
    y_val  = mlb.transform(df_val["Labels"].apply(lambda x: str(x).split(","))).astype(np.float32)
    y_test = mlb.transform(df_test["Labels"].apply(lambda x: str(x).split(","))).astype(np.float32)

    all_lbls = [l for sub in df_train["Labels"].apply(lambda x: str(x).split(",")) for l in sub]
    train_label_counts = Counter(all_lbls)
    target_count = int(round(np.mean(list(train_label_counts.values()))))
    print(f"  Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,} | "
          f"Classes: {len(labels_list)} | Target/class: {target_count}")

    cache_dir = data_base
    protected_set, synonym_dict = build_cyber_knowledge_base(cache_dir=cache_dir)

    configs = build_configs(ca, ga)

    ckpt_path   = out_dir / f"checkpoint_{target_dataset}.json"
    all_results = load_checkpoint(ckpt_path)
    done_ids    = {r["config_id"] for r in all_results}
    proxy_cfg  = {**PROXY_CONFIG, "seed": seed}

    for cfg in configs:
        cid = cfg["id"]
        if cid in done_ids:
            print(f"  [SKIP] {cid} — already in checkpoint")
            continue

        t0 = time.time()
        print(f"\n  [{cid}] group={cfg['group']} sr={cfg['sr']} ri={cfg['ri']} rs={cfg['rs']} rd={cfg['rd']}", flush=True)

        if cfg["eda_type"] is None:
            df_aug  = df_train.copy()
            if "is_augmented" not in df_aug.columns:
                df_aug["is_augmented"] = 0
            n_added = 0
        else:
            df_aug  = generate_augmented_dataset(
                df_train, cfg, protected_set, synonym_dict, target_count, seed=seed)
            n_added = len(df_aug) - len(df_train)
        print(f"    +{n_added:,} samples | Total: {len(df_aug):,}", flush=True)

        vm, tm, vt, tt, zv, zt = train_model(
            df_aug, df_val, df_test, y_val, y_test,
            mlb, train_label_counts, labels_list, device, proxy_cfg)

        elapsed = time.time() - t0
        bv_ref  = all_results[0]["val_metrics"]["Macro_F1"] if all_results else None
        dv_str  = f" (D={vm['Macro_F1']-bv_ref:+.4f})" if bv_ref else ""
        print(f"    Val F1={vm['Macro_F1']:.4f}{dv_str} | Test F1={tm['Macro_F1']:.4f} | {elapsed:.0f}s", flush=True)

        all_results.append({
            "config_id":              cid,
            "label":                  cfg["label"],
            "group":                  cfg["group"],
            "eda_type":               cfg["eda_type"] or "none",
            "alpha_sr":               cfg["sr"],
            "alpha_ri":               cfg["ri"],
            "alpha_rs":               cfg["rs"],
            "alpha_rd":               cfg["rd"],
            "n_augmented_added":      n_added,
            "elapsed_seconds":        round(elapsed, 1),
            "val_metrics":            vm,
            "val_tier_summary":       vt,
            "num_zero_f1_val":        len(zv),
            "test_metrics":           tm,
            "test_tier_summary":      tt,
            "num_zero_f1_test":       len(zt),
        })
        save_checkpoint(ckpt_path, all_results)

    print_summary_table(all_results)

    # -- CSV ------------------------------------------------------------------
    bv = all_results[0]["val_metrics"]["Macro_F1"]
    bt = all_results[0]["test_metrics"]["Macro_F1"]
    rows = []
    for r in all_results:
        vt2, tt2 = r["val_tier_summary"], r["test_tier_summary"]
        rows.append({
            "Config ID":         r["config_id"],
            "Group":             r["group"],
            "EDA Type":          r["eda_type"],
            "SR Alpha":          r["alpha_sr"],
            "RI Alpha":          r["alpha_ri"],
            "RS Alpha":          r["alpha_rs"],
            "RD Alpha":          r["alpha_rd"],
            "N Augmented":       r["n_augmented_added"],
            "Val Macro-F1":      round(r["val_metrics"]["Macro_F1"],  4),
            "Test Macro-F1":     round(r["test_metrics"]["Macro_F1"], 4),
            "Val Macro-Recall":  round(r["val_metrics"]["Macro_Recall"],  4),
            "Test Macro-Recall": round(r["test_metrics"]["Macro_Recall"], 4),
            "Val Micro-F1":      round(r["val_metrics"]["Micro_F1"],  4),
            "Test Micro-F1":     round(r["test_metrics"]["Micro_F1"], 4),
            "Delta Val F1":      round(r["val_metrics"]["Macro_F1"]  - bv, 4),
            "Delta Test F1":     round(r["test_metrics"]["Macro_F1"] - bt, 4),
            "Val Rare F1":       round(vt2.get("Tail (<30)", {}).get("f1_score", 0),  4),
            "Test Rare F1":      round(tt2.get("Tail (<30)", {}).get("f1_score", 0), 4),
            "Val Medium F1":     round(vt2.get("Medium (30-99)", {}).get("f1_score", 0),  4),
            "Test Medium F1":    round(tt2.get("Medium (30-99)", {}).get("f1_score", 0), 4),
            "Val Frequent F1":   round(vt2.get("Major (100-499)", {}).get("f1_score", 0),  4),
            "Test Frequent F1":  round(tt2.get("Major (100-499)", {}).get("f1_score", 0), 4),
            "ZeroF1 Val":        r["num_zero_f1_val"],
            "ZeroF1 Test":       r["num_zero_f1_test"],
            "Time (s)":          r["elapsed_seconds"],
        })
    csv_path = out_dir / f"operation_ablation_results_{target_dataset}.csv"
    pd.DataFrame(rows).to_csv(csv_path, index=False, encoding="utf-8")
    print(f"\n  [SAVED] {csv_path.name}")

    # -- JSON summary ---------------------------------------------------------
    cyber_results   = [r for r in all_results if r["group"] == "CyberEDA"]
    generic_results = [r for r in all_results if r["group"] == "GenericEDA"]
    best_cyber      = max(cyber_results,   key=lambda r: r["val_metrics"]["Macro_F1"])
    best_generic    = max(generic_results, key=lambda r: r["val_metrics"]["Macro_F1"])

    summary = {
        "proxy_model":      "nreimers/MiniLM-L6-H384-uncased",
        "dataset":          target_dataset,
        "seed":             seed,
        "optimal_alpha_cyber":   ca,
        "optimal_alpha_generic": ga,
        "baseline_val_macro_f1":  round(bv, 4),
        "baseline_test_macro_f1": round(bt, 4),
        "best_cyber_config": {
            "id":           best_cyber["config_id"],
            "val_macro_f1": round(best_cyber["val_metrics"]["Macro_F1"], 4),
            "test_macro_f1":round(best_cyber["test_metrics"]["Macro_F1"], 4),
            "delta_val":    round(best_cyber["val_metrics"]["Macro_F1"] - bv, 4),
        },
        "best_generic_config": {
            "id":           best_generic["config_id"],
            "val_macro_f1": round(best_generic["val_metrics"]["Macro_F1"], 4),
            "test_macro_f1":round(best_generic["test_metrics"]["Macro_F1"], 4),
            "delta_val":    round(best_generic["val_metrics"]["Macro_F1"] - bv, 4),
        },
        "operation_ranking_by_delta_val": sorted(
            [{"id": r["config_id"], "group": r["group"],
              "delta_val": round(r["val_metrics"]["Macro_F1"] - bv, 4)}
             for r in all_results if r["group"] != "Baseline"],
            key=lambda x: x["delta_val"], reverse=True),
    }
    json_path = out_dir / f"operation_ablation_summary_{target_dataset}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    print(f"  [SAVED] {json_path.name}")

    # -- Figures --------------------------------------------------------------
    print("\n  Generating figures...")
    plot_macro_f1_comparison(all_results, out_dir, dataset_name=target_dataset)
    plot_delta_f1(all_results, out_dir, dataset_name=target_dataset)
    plot_tail_f1(all_results, out_dir, dataset_name=target_dataset)

    print(f"\n{'='*80}")
    print(f"  OPERATION ABLATION DONE. Best Cyber: {best_cyber['config_id']} | "
          f"Best Generic: {best_generic['config_id']}")
    print(f"{'='*80}")

## Execution


In [ ]:
# Cell 5: Execute Study
# Ensure you have run the ratio ablations first to find optimal alphas.
# If you know the optimal alphas, pass them explicitly below.
run_operation_ablation(target_dataset='joint', seed=42)
